Please note that on 17th of March 2026 the server was down and as of 18th March data prior to 2026 has been removed from the api. This may return at some point.

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
import os

pd.set_option('display.max_columns', None)


folder_name = "Datasets"

try:
    script_dir = os.path.dirname(os.path.abspath(__file__)) # File path where script is running
    folder = os.path.join(script_dir, folder_name)

except Exception:
    # __file__ does not exist in colab
    script_dir = f"."
    folder = os.path.join(script_dir, folder_name)

api_key = "YOUR_API_KEY_HERE"

os.makedirs(folder, exist_ok=True)


In [5]:
def extract_metrics(data):
    rows = []

    for service in data["Services"]:
        attrs = service["serviceAttributesMetrics"]

        origin = attrs["origin_location"]
        destination = attrs["destination_location"]
        gbtt_ptd = attrs["gbtt_ptd"]
        gbtt_pta = attrs["gbtt_pta"]
        toc_code = attrs["toc_code"]

        for rid in attrs["rids"]:
            rows.append({
                "rid": rid,
                "date": rid[:8],
                "origin": origin,
                "destination": destination,
                "gbtt_ptd": gbtt_ptd,
                "gbtt_pta": gbtt_pta,
                "toc_code": toc_code
            })

    return rows

In [20]:
def get_journey_rids(year,
                     from_loc="KGX",
                     to_loc="EDB",
                     from_time="1600",
                     to_time="1900",
                     days="WEEKDAY",
                     range=7,
                     max_retries=10):

    url = "https://api1.raildata.org.uk/1010-historical-service-performance-_hsp_v1/api/v1/serviceMetrics"
    headers = {
        "x-apikey": api_key,
        "User-Agent": "Mozilla/5.0",
        "Content-Type": "application/json"
    }

    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    delta = timedelta(days=range)

    # Build list of date ranges
    date_ranges = []
    current_start = start_date

    while current_start < end_date:
        current_end = min(current_start + delta - timedelta(days=1), end_date)
        date_ranges.append((current_start, current_end))
        current_start += delta

    all_rows = []
    error_df = pd.DataFrame(columns=["from_date","to_date","status_code","error_text"])
    retry_count = 0
    failed_ranges = date_ranges.copy()
    first_run = True

    print("Starting RID retrieval...")

    while failed_ranges and retry_count <= max_retries:
        
        if first_run:
            print(f"\nProcessing {len(failed_ranges)} date ranges...")
            first_run = False
        else:
            print(f"\nRetry round {retry_count} — {len(failed_ranges)} ranges remaining")

        new_failed_ranges = []

        for start, end in failed_ranges:

            body = {
                "from_loc": from_loc,
                "to_loc": to_loc,
                "from_time": from_time,
                "to_time": to_time,
                "from_date": start.strftime("%Y-%m-%d"),
                "to_date": end.strftime("%Y-%m-%d"),
                "days": days
            }

            try:
                response = requests.post(url, json=body, headers=headers, timeout=7200)
            except Exception as e:
                print(f"Request error for {body['from_date']} to {body['to_date']}: {e}")
                new_failed_ranges.append((start, end))
                continue

            if response.status_code == 200:
                data = response.json()
                rows = extract_metrics(data)
                all_rows.extend(rows)
                print(f"✓ {body['from_date']} to {body['to_date']} — {len(rows)} rows")
            else:
                print(f"X Failed {body['from_date']} to {body['to_date']} - Status Code:{response.status_code}")
                new_failed_ranges.append((start, end))

                error_df = pd.concat([error_df, pd.DataFrame([{
                    "from_date": body['from_date'],
                    "to_date": body['to_date'],
                    "status_code": response.status_code,
                    "error_text": response.text
                }])], ignore_index=True)

            time.sleep(2)

        failed_ranges = new_failed_ranges
        retry_count += 1

    print(f"RID retrieval complete. Remaining failures: {len(failed_ranges)}")

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

    return df, error_df

In [9]:
def hhmm_to_minutes(x):
    try:
        x = str(x).zfill(4)  # ensure it’s 4 digits, e.g., '800' -> '0800'
        hours = int(x[:2])
        minutes = int(x[2:])
        return hours * 60 + minutes
    except Exception:
        return np.nan  # handle missing or invalid values


In [19]:
def get_segment_service_details(year, origin = "KGX", destination = "EDB", max_retries = 10, limit = None):

    df = pd.read_csv(os.path.join(folder, f"Evening-Peak-time-RIDs-{year}.csv"))
    df = df[(df["origin"] == origin) & (df["destination"] == destination)]

    url = "https://api1.raildata.org.uk/1010-historical-service-performance-_hsp_v1/api/v1/serviceDetails"

    headers = {
        "x-apikey": api_key,
        "User-Agent": "Mozilla/5.0",
        "Content-Type": "application/json"
    }


    rid_list = df["rid"].tolist()
    if limit is not None:
        rid_list = rid_list[:limit]
    all_rows = []
    failed_rids = []
    first_run = True
    retry_count = 0




    while ((len(failed_rids) > 0 or first_run) and retry_count <= max_retries):

        if not first_run:
            print(f"Retrying {len(failed_rids)} failed RIDs...")
            rid_list = failed_rids
            failed_rids = []
            retry_count += 1
            print(f"Retry attempt {retry_count} of {max_retries}")

        remaining_rids = len(rid_list)

        first_run = False
        for rid in rid_list:
            time.sleep(1.6)  # prevent API rate limiting

            # Send RID to API
            response = requests.post(
                url,
                json={"rid": rid},
                headers=headers
            )

            data = response.json()



            if "serviceAttributesDetails" not in data:
                print("Unexpected response for RID:", rid)
                failed_rids.append(rid)
                remaining_rids -= 1
                print(data)
                continue


            details = data["serviceAttributesDetails"]
            locations = details["locations"]

            total_stops = len(locations)

            for i in range(len(locations) - 1):

                from_stop = locations[i]
                to_stop = locations[i + 1]

                remaining_stops = total_stops - (i + 1)

                row = {
                    "rid": rid,

                    "from_station": from_stop["location"],
                    "to_station": to_stop["location"],

                    # Times
                    "from_sched_arrival": from_stop.get("gbtt_pta"),
                    "from_actual_arrival": from_stop.get("actual_ta"),
                    "from_sched_departure": from_stop.get("gbtt_ptd"),
                    "from_actual_departure": from_stop.get("actual_td"),

                    "to_sched_arrival": to_stop.get("gbtt_pta"),
                    "to_actual_arrival": to_stop.get("actual_ta"),
                    "to_sched_departure": to_stop.get("gbtt_ptd"),
                    "to_actual_departure": to_stop.get("actual_td"),
                    "late_canc_reason": to_stop.get("late_canc_reason", ""),

                    # Stops info
                    "total_stops_in_run": total_stops,
                    "remaining_stops_after_this": remaining_stops
                }

                all_rows.append(row)

            remaining_rids -= 1
            print("Appended RID: ", rid, "\tRemaining RIDs: ", remaining_rids)

    print(f"Finished processing. Total rows: {len(all_rows)}. Failed RIDs: {len(failed_rids)}")

    return pd.DataFrame(all_rows), failed_rids


In [29]:
def create_columns(df, historical = False):

    # Create a tuple column representing each segment
    df['segment_pair'] = list(zip(df['from_station'], df['to_station']))

    # segment_id
    df['segment_id'] = pd.factorize(df['segment_pair'])[0]

    # Extract the date portion from RID (first 8 digits)
    df['date_of_service'] = pd.to_datetime(df['rid'].astype(str).str[:8], format='%Y%m%d')

    # Month (1-12)
    df['month'] = df['date_of_service'].dt.month

    # Day of week (Monday=0, Sunday=6)
    df['day_of_week'] = df['date_of_service'].dt.dayofweek

    # Day of month (1-31)
    df['day_of_month'] = df['date_of_service'].dt.day

    time_cols = [
    'from_sched_arrival', 'from_actual_arrival',
    'from_sched_departure', 'from_actual_departure',
    'to_sched_arrival', 'to_actual_arrival',
    'to_sched_departure', 'to_actual_departure'
    ]

    for col in time_cols:
        df[col + '_min'] = df[col].apply(hhmm_to_minutes)



    #  Propagated delay (delay entering segment)
    df['propagated_delay_min'] = (
        df['from_actual_departure_min'] -
        df['from_sched_departure_min']
    )

    # End-of-segment delay (arrival delay at to_station)
    df['end_segment_delay_min'] = (
        df['to_actual_arrival_min'] -
        df['to_sched_arrival_min']
    )

    # Cumulative delay per train
    df = df.sort_values(by=['rid','from_sched_departure_min'])
    df['cum_delay_min'] = df.groupby('rid')['propagated_delay_min'].cumsum()

    # Leg duration
    df['leg_duration_min'] = (
        df['to_actual_arrival_min'] -
        df['from_actual_departure_min']
    )
    
    #  Arrival delay at station
 
    df["arrival_delay_min"] = (
        df["to_actual_arrival_min"]
        - df["to_sched_arrival_min"]
    )

    if not historical:
        
        
        # End-of-segment delay (arrival delay at to_station)
        df['end_segment_delay_min'] = (
        df['to_actual_arrival_min'] -
        df['to_sched_arrival_min']
        )
        
        # Departure datetime (actual)
        df["departure_datetime"] = (
            pd.to_datetime(df["date_of_service"]) +
            pd.to_timedelta(df["from_actual_departure_min"], unit="m")
        )

        # Arrival datetime (scheduled)
        df["arrival_datetime"] = (
            pd.to_datetime(df["date_of_service"]) +
            pd.to_timedelta(df["to_sched_arrival_min"], unit="m")
        )


        df["departure_hour"] = df["departure_datetime"].dt.hour
        df["arrival_hour"] = df["arrival_datetime"].dt.hour

        # 5. Sort by segment chronologically
        df = df.sort_values(
            by=['segment_id','departure_datetime']
        )

        # 6. Previous train features (same segment)
        df['prev_train_leg_delay_min'] = (
            df.groupby('segment_id')['propagated_delay_min'].shift(1)
        )

        df['prev_train_cum_delay_min'] = (
            df.groupby('segment_id')['cum_delay_min'].shift(1)
        )

        df['prev_train_segment_duration_min'] = (
            df.groupby('segment_id')['leg_duration_min'].shift(1)
        )

        df['prev_train_end_segment_delay_min'] = (
            df.groupby('segment_id')['end_segment_delay_min'].shift(1)
        )

    df.drop(columns=['segment_pair','segment_id'], inplace=True)

    return df

In [13]:
def average_segments(df):

    average_segments = (
        df
        .groupby(["from_station", "to_station"])
        .agg(
            avg_propagated_delay_min=("propagated_delay_min", "mean"),
            avg_leg_duration_min=("leg_duration_min", "mean"),
            avg_cum_delay_min=("cum_delay_min", "mean"),
            avg_arrival_delay_min=("arrival_delay_min", "mean"),
            num_trains=("rid", "nunique")
        )
        .reset_index()
    )

    return average_segments



In [14]:
def get_weather_data(df, station_coords):


    weather_requests = pd.concat([
    df[["from_station", "date_of_service"]].rename(columns={"from_station": "station"}),
    df[["to_station", "date_of_service"]].rename(columns={"to_station": "station"})
    ]).drop_duplicates()

    weather_requests = weather_requests.merge(
    station_coords,
    on="station",
    how="left"
)

    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

    all_weather = []
    remaining_requests = len(weather_requests)

    for _, row in weather_requests.iterrows():

        station = row["station"]
        lat = row["lat"]
        lon = row["lon"]
        date = row["date_of_service"].strftime("%Y-%m-%d")

        print(f"Pulling weather for {station} on {date}")

        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date,
            "end_date": date,
            "hourly": ["temperature_2m","precipitation","windspeed_10m"]
        }

        response = requests.get(url, params=params)
        data = response.json()

        # Safety check
        if "hourly" not in data:
            print("Weather error:", data)
            continue

        hourly = data["hourly"]

        weather_df = pd.DataFrame({
            "station": station,
            "date_of_service": date,
            "hour": pd.to_datetime(hourly["time"]).hour,
            "temperature": hourly["temperature_2m"],
            "precipitation": hourly["precipitation"],
            "wind_speed": hourly["windspeed_10m"]
        })

        all_weather.append(weather_df)
        print(f"Completed weather pull for {station} on {date}. Remaining: {remaining_requests - 1}")
        remaining_requests -= 1
        time.sleep(0.5)  # polite pause (Open-Meteo usually doesn’t need it)

    # Combine everything
    weather_df = pd.concat(all_weather, ignore_index=True)
    
    if not all_weather:
        print("No weather data retrieved.")
        return pd.DataFrame(columns=["station", "date_of_service", "hour", "temperature", "precipitation", "wind_speed"])
    

    print("Weather download complete.")

    return weather_df



In [15]:
def create_final_dataset(df, historical_averages, weather_df):


     # Rename historical columns

    historical_averages = historical_averages.rename(columns={
        "avg_propagated_delay_min": "historical_avg_propagated_delay_min",
        "avg_leg_duration_min": "historical_avg_leg_duration_min",
        "avg_cum_delay_min": "historical_avg_cum_delay_min",
        "avg_arrival_delay_min": "historical_avg_arrival_delay_min"
    })


    # Drop num_trains column

    historical_averages = historical_averages.drop(columns=["num_trains"])

    # Merge into current dataframe

    df = df.merge(
        historical_averages,
        on=["from_station", "to_station"],
        how="left"
    )

    df["delay_historical_difference"] = (
        df["propagated_delay_min"]
        - df["historical_avg_propagated_delay_min"]
    )

    df["date_of_service"] = df["date_of_service"].dt.strftime("%Y-%m-%d")

    df = df.merge(
        weather_df,
        left_on=["from_station", "date_of_service", "departure_hour"],
        right_on=["station", "date_of_service", "hour"],
        how="left"
    )

    df = df.rename(columns={
        "temperature": "from_temp",
        "precipitation": "from_precip",
        "wind_speed": "from_wind"
    })


    df = df.merge(
        weather_df,
        left_on=["to_station","date_of_service","arrival_hour"],
        right_on=["station","date_of_service","hour"],
        how="left"
    )

    df = df.rename(columns={
        "temperature":"to_temp",
        "precipitation":"to_precip",
        "wind_speed":"to_wind"
    })



    cols_to_drop =["station_x","hour_x","station_y","hour_y","departure_datetime", "arrival_datetime", "departure_hour", "arrival_hour",
                   "from_sched_arrival", "from_actual_arrival","from_sched_departure", "from_actual_departure",
                   "to_sched_arrival", "to_actual_arrival", "to_actual_departure", "to_sched_departure"]

    df = df.drop(columns=cols_to_drop)



    return df


In [24]:
# Times out and fails after around 50 seconds, range 3 is the maximum that seems to work without hitting timeouts 
rid_2024, rid_errors_2024 = get_journey_rids(2024)
rid_2024.to_csv(os.path.join(folder, "Evening-Peak-time-RIDs-2024.csv"), index=False)
rid_errors_2024.to_csv(os.path.join(folder, "Evening-Peak-time-RIDs-Errors-2024.csv"), index=False)

Starting RID retrieval...

Processing 53 date ranges...
✓ 2024-01-01 to 2024-01-07 — 33 rows
✓ 2024-01-08 to 2024-01-14 — 33 rows
✓ 2024-01-15 to 2024-01-21 — 35 rows
✓ 2024-01-22 to 2024-01-28 — 29 rows
✓ 2024-01-29 to 2024-02-04 — 27 rows
✓ 2024-02-05 to 2024-02-11 — 35 rows
✓ 2024-02-12 to 2024-02-18 — 35 rows
✓ 2024-02-19 to 2024-02-25 — 19 rows
✓ 2024-02-26 to 2024-03-03 — 29 rows
✓ 2024-03-04 to 2024-03-10 — 31 rows
✓ 2024-03-11 to 2024-03-17 — 33 rows
✓ 2024-03-18 to 2024-03-24 — 28 rows
✓ 2024-03-25 to 2024-03-31 — 35 rows
✓ 2024-04-01 to 2024-04-07 — 34 rows
✓ 2024-04-08 to 2024-04-14 — 35 rows
✓ 2024-04-15 to 2024-04-21 — 34 rows
✓ 2024-04-22 to 2024-04-28 — 35 rows
✓ 2024-04-29 to 2024-05-05 — 35 rows
✓ 2024-05-06 to 2024-05-12 — 30 rows
✓ 2024-05-13 to 2024-05-19 — 35 rows
✓ 2024-05-20 to 2024-05-26 — 31 rows
✓ 2024-05-27 to 2024-06-02 — 35 rows
✓ 2024-06-03 to 2024-06-09 — 35 rows
✓ 2024-06-10 to 2024-06-16 — 35 rows
✓ 2024-06-17 to 2024-06-23 — 35 rows
✓ 2024-06-24 to 202

In [ ]:
rid_2025, rid_errors_2025 = get_journey_rids(2025)
rid_2025.to_csv(os.path.join(folder, "Evening-Peak-time-RIDs-2025.csv"), index=False)
rid_errors_2025.to_csv(os.path.join(folder, "Evening-Peak-time-RIDs-Errors-2025.csv"), index=False)

Starting RID retrieval...

Processing 52 date ranges...
✓ 2025-01-01 to 2025-01-07 — 34 rows
✓ 2025-01-08 to 2025-01-14 — 35 rows
✓ 2025-01-15 to 2025-01-21 — 35 rows
X Failed 2025-01-22 to 2025-01-28 - Status Code:429
✓ 2025-01-29 to 2025-02-04 — 35 rows
✓ 2025-02-05 to 2025-02-11 — 35 rows
✓ 2025-02-12 to 2025-02-18 — 35 rows
✓ 2025-02-19 to 2025-02-25 — 35 rows
✓ 2025-02-26 to 2025-03-04 — 31 rows
X Failed 2025-03-05 to 2025-03-11 - Status Code:429
✓ 2025-03-12 to 2025-03-18 — 35 rows
✓ 2025-03-19 to 2025-03-25 — 35 rows
X Failed 2025-03-26 to 2025-04-01 - Status Code:429
✓ 2025-04-02 to 2025-04-08 — 32 rows
✓ 2025-04-09 to 2025-04-15 — 35 rows
✓ 2025-04-16 to 2025-04-22 — 35 rows
✓ 2025-04-23 to 2025-04-29 — 35 rows
✓ 2025-04-30 to 2025-05-06 — 35 rows
✓ 2025-05-07 to 2025-05-13 — 35 rows
✓ 2025-05-14 to 2025-05-20 — 35 rows
✓ 2025-05-21 to 2025-05-27 — 35 rows
✓ 2025-05-28 to 2025-06-03 — 33 rows
✓ 2025-06-04 to 2025-06-10 — 35 rows
✓ 2025-06-11 to 2025-06-17 — 35 rows
✓ 2025-06-1

In [25]:
df_2024,_ = get_segment_service_details(2024)
df_2024.to_csv(os.path.join(folder, "segments-dataset-2024.csv"), index=False)

Appended RID:  202401016728789 	Remaining RIDs:  1492
Appended RID:  202401016728837 	Remaining RIDs:  1491
Appended RID:  202401026728837 	Remaining RIDs:  1490
Appended RID:  202401036728837 	Remaining RIDs:  1489
Appended RID:  202401046728837 	Remaining RIDs:  1488
Appended RID:  202401056728837 	Remaining RIDs:  1487
Appended RID:  202401016728855 	Remaining RIDs:  1486
Appended RID:  202401026728855 	Remaining RIDs:  1485
Appended RID:  202401036728855 	Remaining RIDs:  1484
Appended RID:  202401046728855 	Remaining RIDs:  1483
Appended RID:  202401016728859 	Remaining RIDs:  1482
Appended RID:  202401026728859 	Remaining RIDs:  1481
Appended RID:  202401036728859 	Remaining RIDs:  1480
Appended RID:  202401046728859 	Remaining RIDs:  1479
Appended RID:  202401056728859 	Remaining RIDs:  1478
Appended RID:  202401016728885 	Remaining RIDs:  1477
Appended RID:  202401026728885 	Remaining RIDs:  1476
Appended RID:  202401036728885 	Remaining RIDs:  1475
Appended RID:  2024010467288

In [30]:
average_segments_2024 = average_segments(create_columns(df_2024, historical=True))
average_segments_2024.to_csv(os.path.join(folder, "historical-averages-2024.csv"), index=False)

In [31]:
df_2025,_ = get_segment_service_details(2025)

Appended RID:  202501017835779 	Remaining RIDs:  1520
Appended RID:  202501016771574 	Remaining RIDs:  1519
Appended RID:  202501026771574 	Remaining RIDs:  1518
Appended RID:  202501036771574 	Remaining RIDs:  1517
Appended RID:  202501066771574 	Remaining RIDs:  1516
Appended RID:  202501076771574 	Remaining RIDs:  1515
Appended RID:  202501016771577 	Remaining RIDs:  1514
Appended RID:  202501026771577 	Remaining RIDs:  1513
Appended RID:  202501036771577 	Remaining RIDs:  1512
Appended RID:  202501066771577 	Remaining RIDs:  1511
Appended RID:  202501076771577 	Remaining RIDs:  1510
Appended RID:  202501016771580 	Remaining RIDs:  1509
Appended RID:  202501026771580 	Remaining RIDs:  1508
Appended RID:  202501036771580 	Remaining RIDs:  1507
Appended RID:  202501066771580 	Remaining RIDs:  1506
Appended RID:  202501076771580 	Remaining RIDs:  1505
Appended RID:  202501016771583 	Remaining RIDs:  1504
Appended RID:  202501026771583 	Remaining RIDs:  1503
Appended RID:  2025010367715

In [32]:
df_2025 = create_columns(df_2025)
df_2025.to_csv(os.path.join(folder, "segments-dataset-2025.csv"), index=False)

In [33]:
weather_df = get_weather_data(df_2025, pd.read_csv(os.path.join(script_dir, "station_coords.csv")))
weather_df.to_csv(os.path.join(folder, "Weather-data-2025.csv"), index=False)

Pulling weather for KGX on 2025-01-01
Completed weather pull for KGX on 2025-01-01. Remaining: 3594
Pulling weather for KGX on 2025-01-02
Completed weather pull for KGX on 2025-01-02. Remaining: 3593
Pulling weather for KGX on 2025-01-03
Completed weather pull for KGX on 2025-01-03. Remaining: 3592
Pulling weather for KGX on 2025-01-06
Completed weather pull for KGX on 2025-01-06. Remaining: 3591
Pulling weather for KGX on 2025-01-07
Completed weather pull for KGX on 2025-01-07. Remaining: 3590
Pulling weather for KGX on 2025-01-08
Completed weather pull for KGX on 2025-01-08. Remaining: 3589
Pulling weather for KGX on 2025-01-09
Completed weather pull for KGX on 2025-01-09. Remaining: 3588
Pulling weather for KGX on 2025-01-10
Completed weather pull for KGX on 2025-01-10. Remaining: 3587
Pulling weather for KGX on 2025-01-13
Completed weather pull for KGX on 2025-01-13. Remaining: 3586
Pulling weather for KGX on 2025-01-14
Completed weather pull for KGX on 2025-01-14. Remaining: 3585


In [34]:
df_2025 = create_final_dataset(df_2025, average_segments_2024, weather_df)
df_2025.to_csv(os.path.join(folder, "final-segment-dataset-2025.csv"), index=False)
